In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/kagglex/final_used_price.csv
/kaggle/input/kagglex/test.csv
/kaggle/input/kagglex/final_train.csv


In [4]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import category_encoders as ce
from sklearn.preprocessing import RobustScaler

In [8]:
train = pd.read_csv('/kaggle/input/kagglex/final_train.csv')
train.head(2)

,brand,milage,fuel_type,transmission,accident,price,HP,Volume,age
0,Ford,74349,Gasoline,10-Speed A/T,0,11000,375.0,3.5,6
1,BMW,80000,Gasoline,6-Speed M/T,0,8250,300.0,3.0,17


In [10]:
original = pd.read_csv('/kaggle/input/kagglex/final_used_price.csv')
original.head(2)

,brand,milage,fuel_type,transmission,accident,price,HP,Volume,age
0,Ford,51000.0,E85 Flex Fuel,6-Speed A/T,1,10300.0,300.0,3.7,11
1,Hyundai,34742.0,Gasoline,8-Speed Automatic,1,38005.0,NaN,3.8,3


In [11]:
the_train = pd.concat([train, original], ignore_index=True)
the_train.head(2)

,brand,milage,fuel_type,transmission,accident,price,HP,Volume,age
0,Ford,74349.0,Gasoline,10-Speed A/T,0,11000.0,375.0,3.5,6
1,BMW,80000.0,Gasoline,6-Speed M/T,0,8250.0,300.0,3.0,17


In [12]:
len(train), len(original), len(the_train)

(54273, 4009, 58282)

In [14]:
test = pd.read_csv('/kaggle/input/kagglex/test.csv')
test.head(2)

,id,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title
0,54273,Mercedes-Benz,E-Class E 350,2014,73000,Gasoline,302.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,A/T,White,Beige,None reported,Yes
1,54274,Lexus,RX 350 Base,2015,128032,Gasoline,275.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Silver,Black,None reported,Yes


In [16]:
kaggle_ids = test['id']
kaggle_ids

0        54273
1        54274
2        54275
3        54276
4        54277
         ...  
36178    90451
36179    90452
36180    90453
36181    90454
36182    90455
Name: id, Length: 36183, dtype: int64

In [17]:
test.drop(columns=['id','model','ext_col','int_col','clean_title'], inplace=True)
test.head(2)

,brand,model_year,milage,fuel_type,engine,transmission,accident
0,Mercedes-Benz,2014,73000,Gasoline,302.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,A/T,None reported
1,Lexus,2015,128032,Gasoline,275.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,None reported


In [18]:
hp_pattern = '(\d+(?:\.\d+))HP'
def extract_hp(text):
    matches = re.findall(hp_pattern, text)
    return float(matches[0]) if matches else np.nan

vol_pattern = '(\d+(?:\.\d+))L'
def extract_vol(text):
    matches = re.findall(vol_pattern, text)
    return float(matches[0]) if matches else np.nan

test['HP'] = test['engine'].apply(extract_hp)
test['Volume'] = test['engine'].apply(extract_vol)

test.drop(columns=['engine'], inplace=True)

In [19]:
test.head()

,brand,model_year,milage,fuel_type,transmission,accident,HP,Volume
0,Mercedes-Benz,2014,73000,Gasoline,A/T,None reported,302.0,3.5
1,Lexus,2015,128032,Gasoline,8-Speed A/T,None reported,275.0,3.5
2,Mercedes-Benz,2015,51983,Gasoline,7-Speed A/T,None reported,241.0,2.0
3,Land,2018,29500,Gasoline,Transmission w/Dual Shift Mode,At least 1 accident or damage reported,518.0,5.0
4,BMW,2020,90000,Gasoline,8-Speed A/T,At least 1 accident or damage reported,335.0,3.0


In [20]:
test['age'] = 2024 - test['model_year']
test.drop(columns=['model_year'], inplace=True)

In [21]:
test['accident'].value_counts()

accident
None reported                             26598
At least 1 accident or damage reported     9585
Name: count, dtype: int64

In [22]:
acc_pattern = 'None'
def accident(text):
    matches = re.findall(acc_pattern, text)
    if matches:
        return 0
    else:
        return 1

test['accident'] = test['accident'].apply(accident)

In [23]:
test.head(2)

,brand,milage,fuel_type,transmission,accident,HP,Volume,age
0,Mercedes-Benz,73000,Gasoline,A/T,0,302.0,3.5,10
1,Lexus,128032,Gasoline,8-Speed A/T,0,275.0,3.5,9


In [24]:
manual_pattern = 'M/T|Manual|Mt|Fixed'
dual_pattern = 'Dual|At/Mt|2'

def ismanual(text):
    match1 = re.findall(manual_pattern, text)
    if(match1):
        return 1
    match2 = re.findall(dual_pattern, text)
    if(match2):
        return 1
    return 0

def isauto(text):
    match1 = re.findall(manual_pattern, text)
    if(match1):
        return 0
    else:
        return 1

the_train['is_auto'] = the_train['transmission'].apply(isauto)
the_train['is_manual'] = the_train['transmission'].apply(ismanual)

test['is_auto'] = test['transmission'].apply(isauto)
test['is_manual'] = test['transmission'].apply(ismanual)


In [25]:
target_encoder = ce.TargetEncoder(cols=['transmission'])
the_train['transmission'] = target_encoder.fit_transform(the_train['transmission'], the_train['price'])
test['transmission'] = target_encoder.transform(test['transmission'])

In [26]:
the_train.head(2)

,brand,milage,fuel_type,transmission,accident,price,HP,Volume,age,is_auto,is_manual
0,Ford,74349.0,Gasoline,60529.132264,0,11000.0,375.0,3.5,6,1,0
1,BMW,80000.0,Gasoline,34274.547594,0,8250.0,300.0,3.0,17,0,1


In [27]:
test.head(2)

,brand,milage,fuel_type,transmission,accident,HP,Volume,age,is_auto,is_manual
0,Mercedes-Benz,73000,Gasoline,30060.575756,0,302.0,3.5,10,1,0
1,Lexus,128032,Gasoline,52428.890420,0,275.0,3.5,9,1,0


In [28]:
def cap_outliers_price(df, col='price'):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3-Q1
    lower_bound = Q1 - (1.5*IQR)
    upper_bound = Q3 + (1.5*IQR)
    df[col] = df[col].apply(lambda x: lower_bound if x<lower_bound else upper_bound if x>upper_bound else x)
    return df, lower_bound, upper_bound

In [29]:
the_train, lb, ub = cap_outliers_price(the_train)
lb, ub

(-29041.5, 90802.5)

In [30]:
the_train['price'].max(), the_train['price'].min()

(90802.5, 2000.0)

In [31]:
the_train.isnull().sum()

brand              0
milage            55
fuel_type          0
transmission       0
accident           0
price              0
HP              4867
Volume          1002
age                0
is_auto            0
is_manual          0
dtype: int64

In [32]:
milage_median = the_train['milage'].median()
hp_median = the_train['HP'].median()
vol_median = the_train['Volume'].median()

the_train['milage'].fillna(milage_median, inplace=True)
the_train['HP'].fillna(hp_median, inplace=True)
the_train['Volume'].fillna(vol_median, inplace=True)

/tmp/ipykernel_34/3032000244.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  the_train['milage'].fillna(milage_median, inplace=True)
/tmp/ipykernel_34/3032000244.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', t

In [33]:
the_train.isnull().sum()

brand           0
milage          0
fuel_type       0
transmission    0
accident        0
price           0
HP              0
Volume          0
age             0
is_auto         0
is_manual       0
dtype: int64

In [34]:
test.isnull().sum()

brand              0
milage             0
fuel_type          0
transmission       0
accident           0
HP              2606
Volume           405
age                0
is_auto            0
is_manual          0
dtype: int64

In [35]:
hp_median = test['HP'].median()
vol_median = test['Volume'].median()

test['HP'].fillna(hp_median, inplace=True)
test['Volume'].fillna(vol_median, inplace=True)

/tmp/ipykernel_34/4237268345.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test['HP'].fillna(hp_median, inplace=True)
/tmp/ipykernel_34/4237268345.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.

In [36]:
test.isnull().sum()

brand           0
milage          0
fuel_type       0
transmission    0
accident        0
HP              0
Volume          0
age             0
is_auto         0
is_manual       0
dtype: int64

In [37]:
the_train.head(2)

,brand,milage,fuel_type,transmission,accident,price,HP,Volume,age,is_auto,is_manual
0,Ford,74349.0,Gasoline,60529.132264,0,11000.0,375.0,3.5,6,1,0
1,BMW,80000.0,Gasoline,34274.547594,0,8250.0,300.0,3.0,17,0,1


In [38]:
the_train['fuel_type'].value_counts()

fuel_type
Gasoline          52918
Hybrid             1960
E85 Flex Fuel      1618
Diesel             1225
Electric            345
Plug-In Hybrid      216
Name: count, dtype: int64

In [39]:
test['fuel_type'].value_counts()

fuel_type
Gasoline          33033
Hybrid             1112
E85 Flex Fuel      1018
Diesel              671
–                   197
Plug-In Hybrid      148
not supported         4
Name: count, dtype: int64

In [41]:
test['fuel_type'] = test['fuel_type'].apply(lambda x: 'Electric' if x=='–' else 'Electric' if x=='not supported' else x)

In [42]:
test['fuel_type'].value_counts()

fuel_type
Gasoline          33033
Hybrid             1112
E85 Flex Fuel      1018
Diesel              671
Electric            201
Plug-In Hybrid      148
Name: count, dtype: int64

In [43]:
cols = ['brand', 'fuel_type']
the_train = pd.get_dummies(the_train, columns=cols, prefix=cols, dummy_na=False, dtype=float)
test = pd.get_dummies(test, columns=cols, prefix=cols, dummy_na=False, dtype=float)

In [44]:
test.insert(test.columns.get_loc('brand_Jeep') + 1, 'brand_Karma', 0)
test.insert(test.columns.get_loc('brand_Maserati') + 1, 'brand_Maybach', 0)
test.insert(test.columns.get_loc('brand_Nissan') + 1, 'brand_Plymouth', 0)
test.insert(test.columns.get_loc('brand_Plymouth') + 1, 'brand_Polestar', 0)

In [46]:
pd.set_option('display.max_columns', None)
the_train.head(2)

,milage,transmission,accident,price,HP,Volume,age,is_auto,is_manual,brand_Acura,brand_Alfa,brand_Aston,brand_Audi,brand_BMW,brand_Bentley,brand_Bugatti,brand_Buick,brand_Cadillac,brand_Chevrolet,brand_Chrysler,brand_Dodge,brand_FIAT,brand_Ferrari,brand_Ford,brand_GMC,brand_Genesis,brand_Honda,brand_Hummer,brand_Hyundai,brand_INFINITI,brand_Jaguar,brand_Jeep,brand_Karma,brand_Kia,brand_Lamborghini,brand_Land,brand_Lexus,brand_Lincoln,brand_Lotus,brand_Lucid,brand_MINI,brand_Maserati,brand_Maybach,brand_Mazda,brand_McLaren,brand_Mercedes-Benz,brand_Mercury,brand_Mitsubishi,brand_Nissan,brand_Plymouth,brand_Polestar,brand_Pontiac,brand_Porsche,brand_RAM,brand_Rivian,brand_Rolls-Royce,brand_Saab,brand_Saturn,brand_Scion,brand_Subaru,brand_Suzuki,brand_Tesla,brand_Toyota,brand_Volkswagen,brand_Volvo,brand_smart,fuel_type_Diesel,fuel_type_E85 Flex Fuel,fuel_type_Electric,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Plug-In Hybrid
0,74349.0,60529.132264,0,11000.0,375.0,3.5,6,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,80000.0,34274.547594,0,8250.0,300.0,3.0,17,0,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [47]:
test.head(2)

,milage,transmission,accident,HP,Volume,age,is_auto,is_manual,brand_Acura,brand_Alfa,brand_Aston,brand_Audi,brand_BMW,brand_Bentley,brand_Bugatti,brand_Buick,brand_Cadillac,brand_Chevrolet,brand_Chrysler,brand_Dodge,brand_FIAT,brand_Ferrari,brand_Ford,brand_GMC,brand_Genesis,brand_Honda,brand_Hummer,brand_Hyundai,brand_INFINITI,brand_Jaguar,brand_Jeep,brand_Karma,brand_Kia,brand_Lamborghini,brand_Land,brand_Lexus,brand_Lincoln,brand_Lotus,brand_Lucid,brand_MINI,brand_Maserati,brand_Maybach,brand_Mazda,brand_McLaren,brand_Mercedes-Benz,brand_Mercury,brand_Mitsubishi,brand_Nissan,brand_Plymouth,brand_Polestar,brand_Pontiac,brand_Porsche,brand_RAM,brand_Rivian,brand_Rolls-Royce,brand_Saab,brand_Saturn,brand_Scion,brand_Subaru,brand_Suzuki,brand_Tesla,brand_Toyota,brand_Volkswagen,brand_Volvo,brand_smart,fuel_type_Diesel,fuel_type_E85 Flex Fuel,fuel_type_Electric,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Plug-In Hybrid
0,73000,30060.575756,0,302.0,3.5,10,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,128032,52428.890420,0,275.0,3.5,9,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [48]:
cols = ['milage','transmission','HP','Volume','age']
sc = RobustScaler()
train_sc = sc.fit_transform(the_train[cols])
the_train['milage']=train_sc[:, 0]
the_train['transmission']=train_sc[:, 1]
the_train['HP']=train_sc[:, 2]
the_train['Volume']=train_sc[:, 3]
the_train['age']=train_sc[:, 4]

In [49]:
the_train.head(2)

,milage,transmission,accident,price,HP,Volume,age,is_auto,is_manual,brand_Acura,brand_Alfa,brand_Aston,brand_Audi,brand_BMW,brand_Bentley,brand_Bugatti,brand_Buick,brand_Cadillac,brand_Chevrolet,brand_Chrysler,brand_Dodge,brand_FIAT,brand_Ferrari,brand_Ford,brand_GMC,brand_Genesis,brand_Honda,brand_Hummer,brand_Hyundai,brand_INFINITI,brand_Jaguar,brand_Jeep,brand_Karma,brand_Kia,brand_Lamborghini,brand_Land,brand_Lexus,brand_Lincoln,brand_Lotus,brand_Lucid,brand_MINI,brand_Maserati,brand_Maybach,brand_Mazda,brand_McLaren,brand_Mercedes-Benz,brand_Mercury,brand_Mitsubishi,brand_Nissan,brand_Plymouth,brand_Polestar,brand_Pontiac,brand_Porsche,brand_RAM,brand_Rivian,brand_Rolls-Royce,brand_Saab,brand_Saturn,brand_Scion,brand_Subaru,brand_Suzuki,brand_Tesla,brand_Toyota,brand_Volkswagen,brand_Volvo,brand_smart,fuel_type_Diesel,fuel_type_E85 Flex Fuel,fuel_type_Electric,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Plug-In Hybrid
0,0.133082,1.17374,0,11000.0,0.511811,0.0000,-0.285714,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.213523,0.00000,0,8250.0,-0.078740,-0.3125,1.285714,0,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [50]:
test_sc = sc.transform(test[cols])
test['milage']=test_sc[:, 0]
test['transmission']=test_sc[:, 1]
test['HP']=test_sc[:, 2]
test['Volume']=test_sc[:, 3]
test['age']=test_sc[:, 4]

In [51]:
test.head(2)

,milage,transmission,accident,HP,Volume,age,is_auto,is_manual,brand_Acura,brand_Alfa,brand_Aston,brand_Audi,brand_BMW,brand_Bentley,brand_Bugatti,brand_Buick,brand_Cadillac,brand_Chevrolet,brand_Chrysler,brand_Dodge,brand_FIAT,brand_Ferrari,brand_Ford,brand_GMC,brand_Genesis,brand_Honda,brand_Hummer,brand_Hyundai,brand_INFINITI,brand_Jaguar,brand_Jeep,brand_Karma,brand_Kia,brand_Lamborghini,brand_Land,brand_Lexus,brand_Lincoln,brand_Lotus,brand_Lucid,brand_MINI,brand_Maserati,brand_Maybach,brand_Mazda,brand_McLaren,brand_Mercedes-Benz,brand_Mercury,brand_Mitsubishi,brand_Nissan,brand_Plymouth,brand_Polestar,brand_Pontiac,brand_Porsche,brand_RAM,brand_Rivian,brand_Rolls-Royce,brand_Saab,brand_Saturn,brand_Scion,brand_Subaru,brand_Suzuki,brand_Tesla,brand_Toyota,brand_Volkswagen,brand_Volvo,brand_smart,fuel_type_Diesel,fuel_type_E85 Flex Fuel,fuel_type_Electric,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Plug-In Hybrid
0,0.113879,-0.18839,0,-0.062992,0.0,0.285714,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.897253,0.81161,0,-0.275591,0.0,0.142857,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [52]:
target='price'
gluon_automl = TabularPredictor(label=target, eval_metric='root_mean_squared_error').fit(the_train, presets='best_quality')

No path specified. Models will be saved in: "AutogluonModels/ag-20240623_082907"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.10.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Tue Dec 19 13:14:11 UTC 2023
CPU Count:          4
Memory Avail:       29.55 GB / 31.36 GB (94.2%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets o

(_ray_fit pid=703) [1000]	valid_set's rmse: 13943.2
(_ray_fit pid=907) [1000]	valid_set's rmse: 14174


(_dystack pid=489) 	-13921.3443	 = Validation score   (-root_mean_squared_error)
(_dystack pid=489) 	27.87s	 = Training   runtime
(_dystack pid=489) 	3.76s	 = Validation runtime
(_dystack pid=489) Fitting model: LightGBM_BAG_L1 ... Training model for up to 561.37s of the 859.22s of remaining time.
(_dystack pid=489) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (4 workers, per: cpus=1, gpus=0, memory=0.15%)


(_ray_fit pid=1156) [1000]	valid_set's rmse: 14125.5


(_dystack pid=489) 	-13878.0493	 = Validation score   (-root_mean_squared_error)
(_dystack pid=489) 	20.83s	 = Training   runtime
(_dystack pid=489) 	2.25s	 = Validation runtime
(_dystack pid=489) Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 536.59s of the 834.44s of remaining time.
(_dystack pid=489) 	-14573.1522	 = Validation score   (-root_mean_squared_error)
(_dystack pid=489) 	46.61s	 = Training   runtime
(_dystack pid=489) 	2.98s	 = Validation runtime
(_dystack pid=489) Fitting model: CatBoost_BAG_L1 ... Training model for up to 485.83s of the 783.68s of remaining time.
(_dystack pid=489) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (4 workers, per: cpus=1, gpus=0, memory=0.19%)
(_dystack pid=489) 	-13871.5481	 = Validation score   (-root_mean_squared_error)
(_dystack pid=489) 	80.1s	 = Training   runtime
(_dystack pid=489) 	0.2s	 = Validation runtime
(_dystack pid=489) Fitting model: ExtraTreesMSE_BAG_L1 ... Training

In [55]:
x = the_train.drop(columns=['price'])
y = the_train['price']

In [56]:
x

,milage,transmission,accident,HP,Volume,age,is_auto,is_manual,brand_Acura,brand_Alfa,brand_Aston,brand_Audi,brand_BMW,brand_Bentley,brand_Bugatti,brand_Buick,brand_Cadillac,brand_Chevrolet,brand_Chrysler,brand_Dodge,brand_FIAT,brand_Ferrari,brand_Ford,brand_GMC,brand_Genesis,brand_Honda,brand_Hummer,brand_Hyundai,brand_INFINITI,brand_Jaguar,brand_Jeep,brand_Karma,brand_Kia,brand_Lamborghini,brand_Land,brand_Lexus,brand_Lincoln,brand_Lotus,brand_Lucid,brand_MINI,brand_Maserati,brand_Maybach,brand_Mazda,brand_McLaren,brand_Mercedes-Benz,brand_Mercury,brand_Mitsubishi,brand_Nissan,brand_Plymouth,brand_Polestar,brand_Pontiac,brand_Porsche,brand_RAM,brand_Rivian,brand_Rolls-Royce,brand_Saab,brand_Saturn,brand_Scion,brand_Subaru,brand_Suzuki,brand_Tesla,brand_Toyota,brand_Volkswagen,brand_Volvo,brand_smart,fuel_type_Diesel,fuel_type_E85 Flex Fuel,fuel_type_Electric,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Plug-In Hybrid
0,0.133082,1.173740,0,0.511811,0.0000,-0.285714,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.213523,0.000000,0,-0.078740,-0.3125,1.285714,0,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.377096,-0.416022,0,-0.078740,0.4375,1.000000,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,-0.890577,0.546557,0,0.196850,-0.3125,-0.857143,1,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.654804,-0.188390,0,-0.866142,0.1875,2.142857,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58277,0.000000,6.473700,0,0.000000,1.5625,-1.000000,1,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
58278,-0.770107,0.546557,0,0.307087,-0.3125,-0.857143,1,1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
58279,-0.895146,1.132466,0,0.000000,0.0000,-0.857143,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
58280,-0.455516,-0.188390,0,1.102362,0.0000,-0.571429,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [57]:
y

0        11000.0
1         8250.0
2        15000.0
3        63500.0
4         7850.0
          ...   
58277    90802.5
58278    53900.0
58279    90802.5
58280    62999.0
58281    40000.0
Name: price, Length: 58282, dtype: float64

In [58]:
y_pred = gluon_automl.predict(x)

In [61]:
from sklearn.metrics import mean_squared_error
train_rmse = np.sqrt(mean_squared_error(y, y_pred))
train_rmse

11346.309032820758

In [62]:
kaggle_predictions = gluon_automl.predict(test)

In [63]:
sub_df = pd.DataFrame({
    'id':kaggle_ids,
    'price':kaggle_predictions
})
sub_df

,id,price
0,54273,20871.142578
1,54274,22929.753906
2,54275,26838.988281
3,54276,55598.578125
4,54277,34370.800781
...,...,...
36178,90451,58000.734375
36179,90452,10018.360352
36180,90453,10823.953125
36181,90454,55057.878906


In [65]:
sub_df.to_csv('sub_ag.csv', index=False)